# Lab 3.1, Build 1: Measure the Haystack

This notebook measures where whole-document retrieval starts costing you precision and tokens.
You will run the same 8 questions per SAR length bucket two ways: whole-document and passage.
Then you will compare raw investigation reports to the pre-computed fact index.

In [ ]:
import os, json, pathlib
from elasticsearch import Elasticsearch
from ara_metrics import run_queries_with_template, precision_at_k, token_count, context_fit

es = Elasticsearch(os.environ["ES_ENDPOINT"], api_key=os.environ["ES_API_KEY"])

# Load your seeded token budget for the break-bucket test
constraint = json.loads(pathlib.Path("/home/elastic/constraint.json").read_text())
token_budget = constraint["token_budget_per_question"]
print(f"Seeded token budget: {token_budget} tokens per question")

# Load dev questions (public)
dev_sar = json.loads(pathlib.Path("/home/elastic/dev-sets/sar-dev-queries.json").read_text())
dev_inv = json.loads(pathlib.Path("/home/elastic/dev-sets/investigations-dev-queries.json").read_text())
print(f"SAR dev queries: {len(dev_sar)} | Investigation dev queries: {len(dev_inv)}")

## Part A: SAR Haystack Measurement

For each of the 5 length buckets, run 8 questions two ways:
1. **Whole-document** retrieval: `cortex-sar-narratives`, top-3 docs, `semantic_text` on `body`
2. **Passage** retrieval: `cortex-sar-passages`, top-5 passages by `section_id`

Record precision (was the answer in what you sent to the model?) and tokens sent.

In [ ]:
buckets = ["bucket_1k", "bucket_3k", "bucket_8k", "bucket_20k", "bucket_40k"]
sar_results = {}

for bucket in buckets:
    bucket_queries = [q for q in dev_sar if q.get("bucket") == bucket]
    
    # Whole-document retrieval
    whole_doc_template = {
        "query": {"semantic": {"field": "body", "query": "{query_text}"}},
        "size": 3
    }
    whole_results = run_queries_with_template(
        es, "cortex-sar-narratives", whole_doc_template, bucket_queries, k=3
    )
    whole_precision = precision_at_k(whole_results, bucket_queries, k=3, gold_field="relevant_ids")
    # Count tokens: top-3 full documents concatenated
    whole_tokens = sum(token_count(r.get("body", "")) for r in whole_results[:3]) if whole_results else 0
    
    # Passage retrieval
    passage_template = {
        "query": {"semantic": {"field": "body", "query": "{query_text}"}},
        "size": 5
    }
    passage_results = run_queries_with_template(
        es, "cortex-sar-passages", passage_template, bucket_queries, k=5
    )
    passage_precision = precision_at_k(
        passage_results, bucket_queries, k=5, gold_field="relevant_passage_ids"
    )
    
    sar_results[bucket] = {
        "whole_doc_precision": whole_precision,
        "whole_doc_tokens_avg": whole_tokens // max(len(bucket_queries), 1),
        "passage_precision": passage_precision,
    }
    print(f"{bucket}: whole_doc p={whole_precision:.2f} tokens={whole_tokens // max(len(bucket_queries), 1)}, passage p={passage_precision:.2f}")

In [ ]:
# Find the break bucket: smallest bucket where whole-doc precision < 0.75 AND tokens > budget
break_bucket = None
for bucket in buckets:
    r = sar_results[bucket]
    if r["whole_doc_precision"] < 0.75 and r["whole_doc_tokens_avg"] > token_budget:
        break_bucket = bucket
        break

if break_bucket:
    print(f"Break bucket: {break_bucket}")
    print(f"  Precision: {sar_results[break_bucket]['whole_doc_precision']:.2f} (below 0.75)")
    print(f"  Avg tokens: {sar_results[break_bucket]['whole_doc_tokens_avg']} (above {token_budget})")
else:
    print("No break bucket found with both criteria. Check your budget setting.")

## Part B: Pre-computed Facts vs. Raw Reports

Run 8 investigation questions two ways:
1. **Raw reports**: `cortex-investigations`, whole documents
2. **Fact index**: `cortex-investigation-facts`, structured records

In [ ]:
# Raw report retrieval
raw_template = {
    "query": {"semantic": {"field": "body", "query": "{query_text}"}},
    "size": 3
}
raw_results = run_queries_with_template(
    es, "cortex-investigations", raw_template, dev_inv, k=3
)
raw_precision = precision_at_k(raw_results, dev_inv, k=3, gold_field="relevant_ids")
raw_tokens = sum(token_count(r.get("body", "")) for r in raw_results[:3]) if raw_results else 0

# Fact index retrieval
fact_template = {
    "query": {"semantic": {"field": "summary", "query": "{query_text}"}},
    "size": 3
}
fact_results = run_queries_with_template(
    es, "cortex-investigation-facts", fact_template, dev_inv, k=3
)
fact_precision = precision_at_k(fact_results, dev_inv, k=3, gold_field="relevant_ids")
fact_tokens = sum(token_count(r.get("summary", "") + " ".join(r.get("answers_questions", []))) 
                  for r in fact_results[:3]) if fact_results else 0

precision_delta = fact_precision - raw_precision
token_ratio = (raw_tokens - fact_tokens) / max(raw_tokens, 1)

print(f"Raw reports:  precision={raw_precision:.2f}, avg_tokens={raw_tokens // max(len(dev_inv), 1)}")
print(f"Fact index:   precision={fact_precision:.2f}, avg_tokens={fact_tokens // max(len(dev_inv), 1)}")
print(f"Delta: +{precision_delta:.2f} precision, -{token_ratio:.0%} tokens")

In [ ]:
import datetime

results = {
    "challenge": "02-measure-the-haystack",
    "written_at": datetime.datetime.utcnow().isoformat(),
    "sar_by_bucket": sar_results,
    "break_bucket": break_bucket,
    "investigation": {
        "raw_precision": raw_precision,
        "raw_tokens_avg": raw_tokens // max(len(dev_inv), 1),
        "fact_precision": fact_precision,
        "fact_tokens_avg": fact_tokens // max(len(dev_inv), 1),
        "precision_delta": precision_delta,
        "token_reduction_pct": token_ratio,
    }
}

traces_dir = pathlib.Path("/home/elastic/.traces")
traces_dir.mkdir(exist_ok=True)
(traces_dir / "haystack-results.json").write_text(json.dumps(results, indent=2))
print("Saved to /home/elastic/.traces/haystack-results.json")
print("Select Check to continue.")